# FuzeAI Final — Hugging Face GPU endpoint

Loads the full `r7194714R/FuzeAI-Final` Transformers model in 4-bit mode, starts an OpenAI-compatible API, and exposes it using a temporary Cloudflare Quick Tunnel. This notebook is designed for Kaggle or Google Colab.

**Before you start:** enable a GPU accelerator and Internet access. If the Hugging Face repository is private, you need a Hugging Face read token. The token is requested with hidden input and is not written into the notebook.

**Security:** anyone who obtains the temporary tunnel URL may use the model. Do not publish the URL or send sensitive code/secrets. Stop the tunnel and session when finished.

In [ ]:
import os, sys, subprocess, pathlib, time, urllib.request, json, re, platform
from getpass import getpass
from pathlib import Path
try:
    import torch
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
    import torch
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle/Colab settings, restart the runtime, and run again.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
MODEL_ID = 'r7194714R/FuzeAI-Final'
HF_TOKEN = getpass('Hugging Face read token (hidden input): ').strip()
if not HF_TOKEN:
    raise RuntimeError('A Hugging Face token is required for this model repository.')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['FUZEAi_MODEL_ID'] = MODEL_ID
print('Model:', MODEL_ID, '| Token entered without echoing it.')

In [ ]:
# Create the standalone server script in the current runtime.
SERVER_SCRIPT = Path('/kaggle/working/fuzeai_hf_endpoint.py') if Path('/kaggle').exists() else Path('/content/fuzeai_hf_endpoint.py')
SERVER_SCRIPT.write_text("#!/usr/bin/env python3\n\"\"\"\nFuzeAI Hugging Face endpoint for Kaggle or Google Colab.\n\nLoads r7194714R/FuzeAI-Final in 4-bit on a CUDA GPU and serves:\n  GET  /health\n  GET  /v1/models\n  POST /v1/chat/completions\n\nRun:\n  python training/fuzeai/fuzeai_hf_endpoint.py\n\nIf the model repository is private, enter a Hugging Face read token when prompted.\nThe service binds to 127.0.0.1 by default; use cloudflared to expose it temporarily.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.util\nimport json\nimport os\nimport re\nimport secrets\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom getpass import getpass\nfrom typing import Any\n\nMODEL_ID = os.environ.get(\"FUZEAi_MODEL_ID\", \"r7194714R/FuzeAI-Final\")\nHOST = os.environ.get(\"FUZEAi_HOST\", \"127.0.0.1\")\nPORT = int(os.environ.get(\"FUZEAi_PORT\", \"8080\"))\nMAX_NEW_TOKENS = int(os.environ.get(\"FUZEAi_MAX_NEW_TOKENS\", \"2048\"))\n\n\ndef ensure_dependencies() -> None:\n    packages = {\n        \"transformers\": \"transformers>=4.48.0\",\n        \"accelerate\": \"accelerate>=1.2.0\",\n        \"bitsandbytes\": \"bitsandbytes>=0.45.0\",\n        \"fastapi\": \"fastapi>=0.115.0\",\n        \"uvicorn\": \"uvicorn>=0.34.0\",\n        \"huggingface_hub\": \"huggingface_hub>=0.27.0\",\n    }\n    missing = [package for module, package in packages.items()\n               if importlib.util.find_spec(module) is None]\n    if missing:\n        print(\"Installing missing dependencies:\", \", \".join(missing), flush=True)\n        subprocess.check_call([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", *missing])\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=\"Serve FuzeAI over an OpenAI-compatible API.\")\n    parser.add_argument(\"--host\", default=HOST)\n    parser.add_argument(\"--port\", type=int, default=PORT)\n    parser.add_argument(\"--model\", default=MODEL_ID)\n    parser.add_argument(\"--max-new-tokens\", type=int, default=MAX_NEW_TOKENS)\n    args = parser.parse_args()\n\n    ensure_dependencies()\n\n    import torch\n    from fastapi import FastAPI, HTTPException\n    from pydantic import BaseModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n    import uvicorn\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\n            \"No CUDA GPU detected. In Kaggle/Colab, enable a GPU accelerator, \"\n            \"restart the runtime, and rerun this script.\"\n        )\n    if args.max_new_tokens < 1 or args.max_new_tokens > 8192:\n        raise ValueError(\"--max-new-tokens must be between 1 and 8192.\")\n\n    token = os.environ.get(\"HF_TOKEN\", \"\").strip()\n    if not token:\n        token = getpass(\"Hugging Face read token (input hidden; needed for private repo): \").strip()\n    if not token:\n        raise RuntimeError(\"A Hugging Face token is required for a private model repository.\")\n\n    from huggingface_hub import HfApi\n    try:\n        HfApi(token=token).whoami()\n    except Exception as exc:\n        raise RuntimeError(\"Hugging Face token authentication failed.\") from exc\n\n    print(f\"Loading {args.model} in 4-bit mode...\", flush=True)\n    print(\"GPU:\", torch.cuda.get_device_name(0), flush=True)\n    quant_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.float16,\n    )\n    tokenizer = AutoTokenizer.from_pretrained(\n        args.model, token=token, trust_remote_code=False\n    )\n    model = AutoModelForCausalLM.from_pretrained(\n        args.model,\n        token=token,\n        quantization_config=quant_config,\n        torch_dtype=torch.float16,\n        device_map=\"auto\",\n        low_cpu_mem_usage=True,\n        trust_remote_code=False,\n    )\n    model.eval()\n    print(\"Model loaded successfully.\", flush=True)\n\n    app = FastAPI(title=\"FuzeAI OpenAI-compatible endpoint\", docs_url=None, redoc_url=None)\n    generation_lock = threading.Lock()\n\n    class ChatRequest(BaseModel):\n        model: str | None = None\n        messages: list[dict[str, Any]]\n        temperature: float = 0.2\n        top_p: float = 0.9\n        max_tokens: int | None = 1024\n        stream: bool = False\n        tools: list[dict[str, Any]] | None = None\n        tool_choice: Any = None\n        stop: str | list[str] | None = None\n\n    @app.get(\"/health\")\n    def health():\n        return {\"status\": \"ok\", \"model\": args.model}\n\n    @app.get(\"/v1/models\")\n    def models():\n        return {\n            \"object\": \"list\",\n            \"data\": [{\n                \"id\": args.model,\n                \"object\": \"model\",\n                \"created\": int(time.time()),\n                \"owned_by\": \"fuzeai\",\n            }],\n        }\n\n    def text_from_content(content: Any) -> str:\n        if isinstance(content, str):\n            return content\n        if isinstance(content, list):\n            pieces = []\n            for part in content:\n                if isinstance(part, dict) and part.get(\"type\") == \"text\":\n                    pieces.append(str(part.get(\"text\", \"\")))\n            return \"\\n\".join(pieces)\n        return \"\" if content is None else str(content)\n\n    @app.post(\"/v1/chat/completions\")\n    def chat_completions(req: ChatRequest):\n        if not req.messages:\n            raise HTTPException(status_code=400, detail=\"messages must not be empty\")\n        if req.stream:\n            raise HTTPException(\n                status_code=400,\n                detail=\"Streaming is not supported by this endpoint; use stream=false.\",\n            )\n\n        messages = []\n        for item in req.messages:\n            role = str(item.get(\"role\", \"user\"))\n            if role not in {\"system\", \"user\", \"assistant\", \"tool\"}:\n                continue\n            entry = {\"role\": role, \"content\": text_from_content(item.get(\"content\"))}\n            if role == \"tool\" and item.get(\"tool_call_id\"):\n                entry[\"name\"] = str(item[\"tool_call_id\"])\n            if role == \"assistant\" and item.get(\"tool_calls\"):\n                entry[\"tool_calls\"] = item[\"tool_calls\"]\n            messages.append(entry)\n\n        max_tokens = req.max_tokens or 1024\n        max_tokens = max(1, min(int(max_tokens), args.max_new_tokens))\n        temperature = max(0.0, min(float(req.temperature), 2.0))\n        top_p = max(0.05, min(float(req.top_p), 1.0))\n\n        try:\n            try:\n                template_args = {\n                    \"conversation\": messages,\n                    \"tokenize\": True,\n                    \"add_generation_prompt\": True,\n                    \"return_tensors\": \"pt\",\n                    \"return_dict\": True,\n                }\n                if req.tools:\n                    template_args[\"tools\"] = req.tools\n                inputs = tokenizer.apply_chat_template(**template_args)\n            except Exception as exc:\n                if req.tools:\n                    raise HTTPException(\n                        status_code=400,\n                        detail=f\"The tokenizer chat template rejected the supplied tool schema: {exc}\",\n                    )\n                raise\n\n            inputs = {k: v.to(model.device) for k, v in inputs.items()}\n            prompt_tokens = int(inputs[\"input_ids\"].shape[-1])\n            generation_args = {\n                **inputs,\n                \"max_new_tokens\": max_tokens,\n                \"do_sample\": temperature > 0,\n                \"temperature\": max(temperature, 0.01),\n                \"top_p\": top_p,\n                \"pad_token_id\": tokenizer.pad_token_id or tokenizer.eos_token_id,\n                \"eos_token_id\": tokenizer.eos_token_id,\n            }\n            if req.stop:\n                stop_words = [req.stop] if isinstance(req.stop, str) else req.stop\n                stop_ids = []\n                for word in stop_words[:4]:\n                    ids = tokenizer.encode(word, add_special_tokens=False)\n                    if ids:\n                        stop_ids.append(ids)\n                # Text stop strings are also trimmed below; don't pass malformed stopping criteria.\n            with generation_lock, torch.inference_mode():\n                output = model.generate(**generation_args)\n            generated_ids = output[0][prompt_tokens:]\n            completion_tokens = int(generated_ids.shape[-1])\n            generated = tokenizer.decode(generated_ids, skip_special_tokens=False).strip()\n\n            # Qwen-family tool-call templates commonly emit <tool_call>{...}</tool_call>.\n            tool_calls = []\n            for match in re.findall(r\"<tool_call>\\s*(\\{.*?\\})\\s*</tool_call>\", generated, flags=re.S):\n                try:\n                    payload = json.loads(match)\n                    name = payload.get(\"name\")\n                    arguments = payload.get(\"arguments\", {})\n                    if name:\n                        if isinstance(arguments, str):\n                            try:\n                                json.loads(arguments)\n                            except Exception:\n                                arguments = json.dumps({\"input\": arguments})\n                        tool_calls.append({\n                            \"id\": \"call_\" + secrets.token_hex(8),\n                            \"type\": \"function\",\n                            \"function\": {\n                                \"name\": str(name),\n                                \"arguments\": arguments if isinstance(arguments, str) else json.dumps(arguments, ensure_ascii=False),\n                            },\n                        })\n                except Exception:\n                    continue\n\n            if tool_calls:\n                content = re.sub(r\"<tool_call>.*?</tool_call>\", \"\", generated, flags=re.S).strip()\n                finish_reason = \"tool_calls\"\n                message = {\"role\": \"assistant\", \"content\": content or None, \"tool_calls\": tool_calls}\n            else:\n                content = re.sub(r\"</?think>\", \"\", generated).strip()\n                content = re.sub(r\"<\\|im_end\\|>|<\\|endoftext\\|>|<\\|fim_suffix\\|>\", \"\", content).strip()\n                if req.stop:\n                    stop_words = [req.stop] if isinstance(req.stop, str) else req.stop\n                    for word in stop_words:\n                        if word and word in content:\n                            content = content.split(word, 1)[0]\n                finish_reason = \"length\" if completion_tokens >= max_tokens else \"stop\"\n                message = {\"role\": \"assistant\", \"content\": content}\n\n            return {\n                \"id\": \"chatcmpl-\" + secrets.token_hex(12),\n                \"object\": \"chat.completion\",\n                \"created\": int(time.time()),\n                \"model\": args.model,\n                \"choices\": [{\n                    \"index\": 0,\n                    \"message\": message,\n                    \"finish_reason\": finish_reason,\n                }],\n                \"usage\": {\n                    \"prompt_tokens\": prompt_tokens,\n                    \"completion_tokens\": completion_tokens,\n                    \"total_tokens\": prompt_tokens + completion_tokens,\n                },\n            }\n        except HTTPException:\n            raise\n        except Exception as exc:\n            print(f\"Inference request failed: {type(exc).__name__}: {exc}\", flush=True)\n            raise HTTPException(status_code=500, detail=f\"Inference failed: {type(exc).__name__}\")\n\n    print(f\"API ready at http://{args.host}:{args.port}/v1\", flush=True)\n    uvicorn.run(app, host=args.host, port=args.port, log_level=\"info\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
print('Standalone script saved to:', SERVER_SCRIPT)
print('For a reusable copy, download training/fuzeai/fuzeai_hf_endpoint.py from the FuzeCLI repository.')

In [ ]:
# Start the server as a subprocess. Logs are saved locally for debugging.
LOG_PATH = SERVER_SCRIPT.with_name('fuzeai-hf-server.log')
server_log = open(LOG_PATH, 'w', encoding='utf-8')
server = subprocess.Popen([sys.executable, str(SERVER_SCRIPT)], stdout=server_log, stderr=subprocess.STDOUT, env=os.environ.copy())
ready = False
for _ in range(240):
    if server.poll() is not None:
        server_log.flush()
        print(LOG_PATH.read_text(errors='replace')[-12000:])
        raise RuntimeError('FuzeAI server exited before becoming ready.')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=3) as response:
            if response.status == 200:
                print('Health:', response.read().decode())
                ready = True
                break
    except Exception:
        time.sleep(3)
if not ready:
    print(LOG_PATH.read_text(errors='replace')[-12000:])
    raise TimeoutError('Model did not become ready within 12 minutes.')
print('OpenAI-compatible API:', 'http://127.0.0.1:8080/v1')
print('Models endpoint:', 'http://127.0.0.1:8080/v1/models')

In [ ]:
# Smoke-test the model with a real completion before exposing the endpoint.
import urllib.request
payload = {'model': MODEL_ID, 'messages': [{'role':'user','content':'Reply with exactly: FuzeAI endpoint ready.'}], 'max_tokens': 32, 'temperature': 0}
request = urllib.request.Request('http://127.0.0.1:8080/v1/chat/completions', data=json.dumps(payload).encode(), headers={'Content-Type':'application/json'}, method='POST')
with urllib.request.urlopen(request, timeout=180) as response:
    result = json.loads(response.read().decode())
print(result['choices'][0]['message'].get('content', ''))
print('Smoke test passed.')

In [ ]:
# Install cloudflared on x86_64 Linux, then open a temporary HTTPS tunnel.
if platform.machine().lower() not in ('x86_64', 'amd64'):
    raise RuntimeError('This cloudflared install cell currently expects an x86_64 Kaggle/Colab runtime.')
deb = '/tmp/cloudflared.deb'
subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb','-O',deb], check=True)
subprocess.run(['dpkg','-i',deb], check=True, stdout=subprocess.DEVNULL)
print('cloudflared version:', subprocess.run(['cloudflared','--version'],capture_output=True,text=True,check=True).stdout.strip())
print('Starting tunnel. Keep this cell running; copy the https://...trycloudflare.com URL.')
tunnel = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8080'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in tunnel.stdout:
        print(line, end='')
except KeyboardInterrupt:
    tunnel.terminate()
    raise

## Connect your Windows FuzeCLI

Copy the HTTPS URL printed by the tunnel cell. In PowerShell, run:

```powershell
aicli fuzeai connect https://YOUR-URL.trycloudflare.com
aicli fuzeai status
aicli chat
```

FuzeCLI now selects the `r7194714R/FuzeAI-Final` remote model. To restore local GGUF mode, run `aicli fuzeai disconnect`. Keep the notebook and tunnel running while using the remote model. Interrupt the tunnel cell and stop the runtime when finished.

The notebook binds the model API to loopback, but the Quick Tunnel URL makes it reachable publicly. Treat the URL as a secret. Quick Tunnels are for testing, have no uptime guarantee, and are not permanent hosting.